In [3]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np

from dotenv import load_dotenv


# ============================================================
# CONFIGURATION
# ============================================================

load_dotenv()

API_KEY = "f217016c-1b59-4053-b0ea-3a5b4e95d4af"

if not API_KEY:
    raise ValueError(
        "OCM_API_KEY not found. "
        "Put it inside your .env file."
    )


BASE_URL = "https://api.openchargemap.io/v3/poi/"

COUNTRY_CODE = "IN"

RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)


# ============================================================
# API CONFIGURATION
# ============================================================

params = {

    "output": "json",

    "countrycode": COUNTRY_CODE,

    "maxresults": 100000,

    "compact": "false",

    "verbose": "true"

}


headers = {

    "X-API-Key": API_KEY,

    "User-Agent":
        "VoltVisionAI/1.0 EV Infrastructure Research"

}


# ============================================================
# 1. REQUEST DATA
# ============================================================

print("=" * 70)
print("OPEN CHARGE MAP DATA COLLECTION")
print("=" * 70)

print("\nRequesting charging station data for India...")


try:

    response = requests.get(

        BASE_URL,

        params=params,

        headers=headers,

        timeout=120

    )

except requests.RequestException as e:

    raise RuntimeError(
        f"API request failed: {e}"
    )


# ============================================================
# 2. CHECK RESPONSE
# ============================================================

print(
    "\nHTTP Status:",
    response.status_code
)


if response.status_code != 200:

    print("\nAPI Response:")

    print(response.text[:2000])

    raise RuntimeError(
        "Open Charge Map API request failed."
    )


# ============================================================
# 3. PARSE JSON
# ============================================================

try:

    data = response.json()

except Exception:

    print(
        response.text[:2000]
    )

    raise ValueError(
        "API did not return valid JSON."
    )


print(
    "\nCharging locations received:",
    len(data)
)


# ============================================================
# 4. SAVE RAW DATA
# ============================================================

raw_file = (
    f"{RAW_DIR}/"
    "charging_stations_raw.json"
)


with open(
    raw_file,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    "\nRaw data saved:",
    raw_file
)


# ============================================================
# 5. EXTRACT STATIONS
# ============================================================

stations = []


for item in data:

    address = (
        item.get("AddressInfo")
        or {}
    )

    operator = (
        item.get("OperatorInfo")
        or {}
    )

    usage = (
        item.get("UsageType")
        or {}
    )

    status = (
        item.get("StatusType")
        or {}
    )

    connections = (
        item.get("Connections")
        or []
    )


    # --------------------------------------------------------
    # BASIC LOCATION INFORMATION
    # --------------------------------------------------------

    station_id = item.get(
        "ID"
    )

    title = address.get(
        "Title"
    )

    latitude = address.get(
        "Latitude"
    )

    longitude = address.get(
        "Longitude"
    )

    address_line = address.get(
        "AddressLine1"
    )

    town = address.get(
        "Town"
    )

    state = address.get(
        "StateOrProvince"
    )

    postcode = address.get(
        "PostCode"
    )

    country = address.get(
        "Country",
        {}
    )

    country_name = (
        country.get("Title")
        if isinstance(country, dict)
        else None
    )


    # --------------------------------------------------------
    # OPERATOR
    # --------------------------------------------------------

    operator_name = operator.get(
        "Title"
    )


    # --------------------------------------------------------
    # USAGE
    # --------------------------------------------------------

    usage_type = usage.get(
        "Title"
    )


    # --------------------------------------------------------
    # STATUS
    # --------------------------------------------------------

    status_type = status.get(
        "Title"
    )


    # --------------------------------------------------------
    # NUMBER OF CHARGING POINTS
    # --------------------------------------------------------

    number_of_points = item.get(
        "NumberOfPoints"
    )


    # --------------------------------------------------------
    # CONNECTION INFORMATION
    # --------------------------------------------------------

    connection_types = []

    power_values = []

    quantity_values = []

    level_values = []


    for connection in connections:

        if not isinstance(
            connection,
            dict
        ):
            continue


        # Connection type
        connection_type = (
            connection
            .get("ConnectionType")
            or {}
        )


        connection_title = (
            connection_type
            .get("Title")
        )


        if connection_title:

            connection_types.append(
                connection_title
            )


        # Power
        power_kw = connection.get(
            "PowerKW"
        )


        if power_kw is not None:

            try:

                power_values.append(
                    float(power_kw)
                )

            except (
                ValueError,
                TypeError
            ):

                pass


        # Quantity
        quantity = connection.get(
            "Quantity"
        )


        if quantity is not None:

            try:

                quantity_values.append(
                    float(quantity)
                )

            except (
                ValueError,
                TypeError
            ):

                pass


        # Level
        level = (
            connection
            .get("Level")
            or {}
        )


        level_title = level.get(
            "Title"
        )


        if level_title:

            level_values.append(
                level_title
            )


    # --------------------------------------------------------
    # AGGREGATED CONNECTION DATA
    # --------------------------------------------------------

    if power_values:

        max_power = max(
            power_values
        )

        average_power = np.mean(
            power_values
        )

    else:

        max_power = np.nan

        average_power = np.nan


    if quantity_values:

        total_connections = int(
            sum(quantity_values)
        )

    elif number_of_points:

        try:

            total_connections = int(
                number_of_points
            )

        except (
            ValueError,
            TypeError
        ):

            total_connections = 1

    else:

        # OCM notes that NumberOfPoints/
        # connection information can be
        # incomplete, so default to 1
        # for a station with connection data.

        total_connections = (
            len(connections)
            if connections
            else 0
        )


    stations.append({

        "station_id":
            station_id,

        "station_name":
            title,

        "latitude":
            latitude,

        "longitude":
            longitude,

        "address":
            address_line,

        "city":
            town,

        "state":
            state,

        "postcode":
            postcode,

        "country":
            country_name,

        "operator":
            operator_name,

        "usage_type":
            usage_type,

        "status":
            status_type,

        "number_of_points":
            number_of_points,

        "total_connections":
            total_connections,

        "connection_types":
            " | ".join(
                sorted(
                    set(
                        connection_types
                    )
                )
            ),

        "charging_levels":
            " | ".join(
                sorted(
                    set(
                        level_values
                    )
                )
            ),

        "max_power_kw":
            max_power,

        "average_power_kw":
            average_power

    })


# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

ocm_df = pd.DataFrame(
    stations
)


print(
    "\nRaw station dataframe shape:",
    ocm_df.shape
)


# ============================================================
# 7. CLEAN LATITUDE / LONGITUDE
# ============================================================

ocm_df["latitude"] = pd.to_numeric(
    ocm_df["latitude"],
    errors="coerce"
)

ocm_df["longitude"] = pd.to_numeric(
    ocm_df["longitude"],
    errors="coerce"
)


ocm_df = ocm_df.dropna(
    subset=[
        "latitude",
        "longitude"
    ]
)


# ============================================================
# 8. REMOVE INVALID INDIA COORDINATES
# ============================================================

ocm_df = ocm_df[
    (
        ocm_df["latitude"].between(
            6,
            38
        )
    )
    &
    (
        ocm_df["longitude"].between(
            68,
            98
        )
    )
].copy()


# ============================================================
# 9. REMOVE DUPLICATE STATIONS
# ============================================================

ocm_df = ocm_df.drop_duplicates(
    subset=["station_id"]
)


# ============================================================
# 10. NORMALIZE TEXT
# ============================================================

text_columns = [
    "station_name",
    "city",
    "state",
    "operator",
    "usage_type",
    "status",
    "connection_types",
    "charging_levels"
]


for column in text_columns:

    if column in ocm_df.columns:

        ocm_df[column] = (
            ocm_df[column]
            .fillna("")
            .astype(str)
            .str.strip()
        )


# ============================================================
# 11. CLEAN POWER
# ============================================================

ocm_df["max_power_kw"] = pd.to_numeric(
    ocm_df["max_power_kw"],
    errors="coerce"
)

ocm_df["average_power_kw"] = pd.to_numeric(
    ocm_df["average_power_kw"],
    errors="coerce"
)

ocm_df["total_connections"] = pd.to_numeric(
    ocm_df["total_connections"],
    errors="coerce"
).fillna(0)


# ============================================================
# 12. FAST CHARGER FLAG
# ============================================================

ocm_df["is_fast_charger"] = (
    ocm_df["max_power_kw"] >= 50
).astype(int)


# ============================================================
# 13. HIGH POWER CHARGER FLAG
# ============================================================

ocm_df["is_high_power"] = (
    ocm_df["max_power_kw"] >= 100
).astype(int)


# ============================================================
# 14. OPERATIONAL FLAG
# ============================================================

ocm_df["is_operational"] = (
    ocm_df["status"]
    .str.upper()
    .str.contains(
        "OPERATIONAL",
        na=False
    )
).astype(int)


# ============================================================
# 15. PUBLIC STATION FLAG
# ============================================================

ocm_df["is_public"] = (
    ocm_df["usage_type"]
    .str.upper()
    .str.contains(
        "PUBLIC",
        na=False
    )
).astype(int)


# ============================================================
# 16. SAVE CLEAN DATA
# ============================================================

output_file = (
    f"{PROCESSED_DIR}/"
    "charging_stations.csv"
)


ocm_df.to_csv(
    output_file,
    index=False
)


print(
    "\nClean charging station dataset saved:"
)

print(
    output_file
)


# ============================================================
# 17. SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("OPEN CHARGE MAP SUMMARY")
print("=" * 70)


print(
    "\nTotal stations:",
    len(ocm_df)
)


print(
    "Operational stations:",
    ocm_df["is_operational"].sum()
)


print(
    "Public stations:",
    ocm_df["is_public"].sum()
)


print(
    "Fast charger stations:",
    ocm_df["is_fast_charger"].sum()
)


print(
    "High power stations:",
    ocm_df["is_high_power"].sum()
)


print(
    "Total charging connections:",
    int(
        ocm_df["total_connections"].sum()
    )
)


print(
    "\nStates:"
)

print(
    ocm_df["state"]
    .value_counts()
    .head(20)
)


# ============================================================
# 18. POWER SUMMARY
# ============================================================

print("\nPower statistics:")

print(
    ocm_df[
        [
            "max_power_kw",
            "average_power_kw"
        ]
    ].describe()
)


# ============================================================
# 19. SHOW SAMPLE
# ============================================================

print("\nFirst 10 stations:")

print(
    ocm_df[
        [
            "station_id",
            "station_name",
            "city",
            "state",
            "latitude",
            "longitude",
            "operator",
            "total_connections",
            "max_power_kw",
            "status"
        ]
    ].head(10)
)


print("\n" + "=" * 70)
print("OPEN CHARGE MAP PIPELINE COMPLETED")
print("=" * 70)

OPEN CHARGE MAP DATA COLLECTION

Requesting charging station data for India...

HTTP Status: 200

Charging locations received: 1949

Raw data saved: data/raw/charging_stations_raw.json

Raw station dataframe shape: (1949, 18)

Clean charging station dataset saved:
data/processed/charging_stations.csv

OPEN CHARGE MAP SUMMARY

Total stations: 1947
Operational stations: 1928
Public stations: 1909
Fast charger stations: 682
High power stations: 110
Total charging connections: 4906

States:
state
Kerala             591
Odisha             370
Karnataka          366
Tamil Nadu         204
Maharashtra         78
Gujarat             64
Rajasthan           46
Delhi               32
Haryana             28
Uttar Pradesh       26
Telangana           16
                    13
Uttarakhand         12
Bangalore Urban     10
Madhya Pradesh       8
Andhra Pradesh       7
West Bengal          7
Goa                  6
Tamilnadu            5
MP                   5
Name: count, dtype: int64

Power statistic